# 03 - Temporal Split

**Day 1 - Cross-Temporal Hybrid NIDS**

This notebook demonstrates the **most important Day 1 research
methodology**: models are trained on historical traffic and evaluated on
strictly *later* traffic - never on a random shuffle.

```text
Historical traffic
       |
       v
Training data
       |
       v
  TIME BOUNDARY
       |
       v
Later traffic
       |
       v
 Test data
```

All splitting/verification logic lives in `src/data/temporal_split.py`;
this notebook only calls it and visualizes the result.

> **Status:** this notebook has **not** been executed against the real
> dataset. It reads a single bounded chunk for demonstration - the
> production train/test artifacts are produced by
> `scripts/prepare_data.py`, which performs the split over the full
> cleaned dataset (see the memory-strategy note in that script).

In [1]:
# --- Setup -----------------------------------------------------------
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data.loader import discover_dataset_files, iter_dataset_chunks, DEFAULT_CHUNK_SIZE
from src.data.validator import validate_schema
from src.data.temporal_split import (
    detect_timestamp_column,
    chronological_train_test_split,
    verify_no_temporal_leakage,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "CIC-IDS2017"
RAW_DIR

WindowsPath('c:/Users/abuba/Downloads/cross-temporal-hybrid-nids-day1/cross-temporal-hybrid-nids/data/raw/CIC-IDS2017')

## 1. Load one chunk and detect the timestamp column

Reusing the same detection utility from `01_dataset_inspection.ipynb`.
A correct chronological split is impossible without a trustworthy
timestamp column, so this step is never skipped.

In [5]:
chunk = next(
    iter_dataset_chunks(
        RAW_DIR,
        chunk_size=DEFAULT_CHUNK_SIZE
    )
)

schema_report = validate_schema(chunk)

ts_result = detect_timestamp_column(chunk)

print(f"Detected timestamp column: {ts_result.column!r}")
print(f"Parsed successfully      : {ts_result.parsed_successfully}")
print(f"Unparseable values       : {ts_result.n_unparseable}")

assert ts_result.column is not None, (
    "No timestamp column detected - pass an explicit column name to "
    "detect_timestamp_column(chunk, explicit_column=...) once you know it."
)

timestamp_col = ts_result.column

No timestamp-like column detected by name. Chronological splitting cannot proceed without an explicit column.


Detected timestamp column: None
Parsed successfully      : False
Unparseable values       : 0


AssertionError: No timestamp column detected - pass an explicit column name to detect_timestamp_column(chunk, explicit_column=...) once you know it.

## 2. Timestamp distribution

A quick look at how flow timestamps are distributed across this chunk,
before any splitting happens.

In [ ]:
parsed_ts = pd.to_datetime(chunk[timestamp_col], errors="coerce")
print(parsed_ts.describe())

fig, ax = plt.subplots(figsize=(9, 4))
parsed_ts.dropna().hist(bins=50, ax=ax)
ax.set_title(f"Timestamp distribution ({timestamp_col})")
ax.set_xlabel("time")
ax.set_ylabel("row count")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Chronological ordering

Sorting explicitly by timestamp (mergesort for a stable, reproducible
order) - this is the ordering the split itself relies on.

In [ ]:
ordered = chunk.assign(_ts=parsed_ts).sort_values("_ts", kind="mergesort")
print("Earliest rows:")
display(ordered[[timestamp_col]].head(3))
print("Latest rows:")
display(ordered[[timestamp_col]].tail(3))

## 4. Chronological train/test split

`chronological_train_test_split` sorts by the timestamp column and
places the chronologically **latest** `test_size` fraction of rows into
the test set - no shuffling, no `random_state`, no stratified sampling.

**Do not** substitute this with `sklearn.model_selection.train_test_split`
(random shuffling) as the primary experiment - that would leak future
traffic patterns into training and invalidate the "cross-temporal"
premise of this project. A random split is deliberately *not*
demonstrated here as anything other than an anti-pattern (see the
commented-out cell below).

In [ ]:
split = chronological_train_test_split(chunk, timestamp_column=timestamp_col, test_size=0.2)
split.summary()

In [ ]:
# --- ANTI-PATTERN (do NOT use as the primary experiment) -------------
# A random split would look like this - shown ONLY to contrast with the
# chronological split above. It is intentionally left unexecuted/unused
# in this project because it violates the cross-temporal evaluation
# design (train and test rows could come from the same or even reversed
# time windows).
#
# from sklearn.model_selection import train_test_split
# train_random, test_random = train_test_split(
#     chunk, test_size=0.2, random_state=42, shuffle=True
# )

## 5. Train/test sizes and earliest/latest timestamps

In [ ]:
summary = split.summary()
print(f"Train rows  : {summary['train_rows']}")
print(f"Test rows   : {summary['test_rows']}")
print(f"Train window: {summary['train_start']}  ->  {summary['train_end']}")
print(f"Test window : {summary['test_start']}  ->  {summary['test_end']}")
print(f"Split point : {summary['split_timestamp']}")

## 6. Explicit temporal leakage check

This is a **hard gate**, not a warning: `verify_no_temporal_leakage`
checks that the training window ends at or before the test window
begins, and that no individual timestamp value appears on both sides.
`scripts/prepare_data.py` raises and aborts if this check fails - it
never silently continues.

In [ ]:
leakage_check = verify_no_temporal_leakage(split)

print(leakage_check.message)
print(f"passed: {leakage_check.passed}")

assert leakage_check.passed, 'Temporal leakage check failed - do not proceed to training.'

## 7. Visualization of the split

Train rows plotted in one color, test rows in another, along the time
axis, with the split boundary marked - a visual confirmation that train
is strictly historical and test is strictly later.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

train_ts = pd.to_datetime(split.train_df[timestamp_col], errors="coerce")
test_ts = pd.to_datetime(split.test_df[timestamp_col], errors="coerce")

ax.scatter(train_ts, [0] * len(train_ts), s=4, label="train (historical)", alpha=0.5)
ax.scatter(test_ts, [0] * len(test_ts), s=4, label="test (later)", alpha=0.5)
ax.axvline(split.split_timestamp, color="black", linestyle="--", label="split boundary")

ax.set_yticks([])
ax.set_xlabel("time")
ax.set_title("Chronological train/test split")
ax.legend(loc="upper left")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Summary

* Timestamp column detected and its distribution inspected.
* Data explicitly sorted chronologically (not shuffled).
* `chronological_train_test_split` used as the **primary** splitting
  method - a random split was shown only as a labeled anti-pattern.
* `verify_no_temporal_leakage` confirmed the split is valid (hard gate).
* The split visualized along the time axis.

Next: `04_baseline_model.ipynb` for the Random Forest baseline trained
on this chronological split.